# Controlled ResNet50 fold-0 pilot — Drive input, preparation only
Same r2 dataset ZIP, hashes, independent ImageNet initialization and fixed 1+5+5 budget.
544 fit /136 score from original Training only; labels provisional, participant independence
unverified. No Test, held records, personal photos or previous dental checkpoints.

After explicit cloud permission, manually place `resnet50_fold0_pilot_v1.zip` in
`My Drive/RP-J26-IT-458/roi_tooth_v1/` (the same folder as the older ROI files).
Use the355MB r2 ZIP with SHA starting `60f70f27`, not a previous failed preparation.
The mount cell asks you to authorize Drive access. The ZIP is hash-checked then copied to
the runtime; extraction and training remain on local VM storage. No automatic image uploader,
Drive result writer, extra training, changed inputs or demo integration is introduced.

Saved flags remain false: preparation does not grant training permission. Keep notebook and
outputs private. Same experimental/clinical limitations as r2. A Drive input does not repair
an already loaded mixed NumPy installation; restart the session when the setup cell requests it.


In [ ]:
# Keep both False during preparation. Run All cannot start training by default.
import os, sys
from pathlib import Path
sys.dont_write_bytecode = True
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
CLOUD_UPLOAD_APPROVED = False
RUN_TRAINING = False
OWNER_APPROVAL = ""
ARCHIVE = Path("/content/resnet50_fold0_drive_r3_input.zip")
DRIVE_FOLDER = Path("/content/drive/MyDrive/RP-J26-IT-458/roi_tooth_v1")
ROOT = Path("/content/resnet50_fold0_package_v1")
RUN_ROOT = Path("/content/resnet50_fold0_drive_r3_run_v1")
ARCHIVE_SHA = "60f70f2703a95c870586190705e991ca72dfa6a2e999096a554585b2433665fa"
MANIFEST_SHA = "1d2a49284a885621aa3b89d059f2be6283f381aa9449db0c4ead8eca206ebec4"


In [ ]:
# Install only mismatched packages; then STOP for a session restart.
if not CLOUD_UPLOAD_APPROVED:
    raise PermissionError("Cloud processing has not been approved. Training stays disabled.")
import importlib.metadata, subprocess
required = {"torch": "2.11.0", "torchvision": "0.26.0", "numpy": "2.5.3",
            "Pillow": "11.3.0", "scikit-learn": "1.9.1"}
missing = []
for name, version in required.items():
    try:
        current = importlib.metadata.version(name).split("+")[0]
    except importlib.metadata.PackageNotFoundError:
        current = None
    if current != version:
        missing.append(name + "==" + version)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    raise RuntimeError("Dependencies installed. Runtime > Restart session; then rerun from the configuration cell.")
loaded_numpy = sys.modules.get("numpy")
if loaded_numpy is not None and getattr(loaded_numpy, "__version__", None) != required["numpy"]:
    raise RuntimeError("Old NumPy is still loaded. Runtime > Restart session; rerun configuration/setup.")
print("Pinned package versions present. No package changes or training in this cell.")


## Mount Drive and copy the already uploaded ZIP
Put the ZIP in the folder above once. Run this cell only after cloud processing permission;
approve the Google authorization dialog yourself. No Drive files are deleted or replaced.
If using a shared-folder shortcut, set DRIVE_FOLDER to its actual My Drive path.


In [ ]:
if not CLOUD_UPLOAD_APPROVED:
    raise PermissionError("Cloud processing has not been approved")
from google.colab import drive
import hashlib, shutil
drive.mount("/content/drive")
def input_sha(path):
    result = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024*1024), b""):
            result.update(block)
    return result.hexdigest()
drive_zip = DRIVE_FOLDER / "resnet50_fold0_pilot_v1.zip"
if not drive_zip.is_file():
    raise FileNotFoundError("Manually upload the r2 ZIP to: " + str(DRIVE_FOLDER))
if input_sha(drive_zip) != ARCHIVE_SHA:
    raise ValueError("Drive ZIP hash mismatch: wait for upload completion and verify the r2 file. Do not change the hash.")
if ARCHIVE.exists():
    if input_sha(ARCHIVE) != ARCHIVE_SHA:
        raise ValueError("Existing local copy differs; preserve it and use a fresh ARCHIVE path.")
else:
    with drive_zip.open("rb") as source, ARCHIVE.open("xb") as target:
        shutil.copyfileobj(source, target, 1024*1024)
    if input_sha(ARCHIVE) != ARCHIVE_SHA:
        raise ValueError("Local copy integrity failed; preserve it and inspect before proceeding")
print("Verified local ZIP:", ARCHIVE)


## Verify/extract the local copy and run data preflight
The following integrity, extraction and preflight code is unchanged from r2.
Drive remains the input archive source only. The training runner remains unchanged.


In [ ]:
if not CLOUD_UPLOAD_APPROVED:
    raise PermissionError("Cloud processing has not been approved")
import hashlib, json, stat, zipfile
def file_sha(path):
    result = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024*1024), b""):
            result.update(block)
    return result.hexdigest()
assert file_sha(ARCHIVE) == ARCHIVE_SHA, "Wrong or damaged ZIP"
if not ROOT.exists():
    with zipfile.ZipFile(ARCHIVE) as archive:
        entries = archive.infolist()
        names = [entry.filename for entry in entries]
        assert len(names) == len(set(names)) < 750
        assert sum(entry.file_size for entry in entries) < 1024**3
        for entry in entries:
            name = Path(entry.filename)
            assert not name.is_absolute() and ".." not in name.parts
            assert not entry.is_dir() and not stat.S_ISLNK(entry.external_attr >> 16)
            assert entry.file_size < 110*1024**2
        assert hashlib.sha256(archive.read("manifest.json")).hexdigest() == MANIFEST_SHA
        manifest = json.loads(archive.read("manifest.json"))
        assert set(names) == set(manifest["files"]) | {"manifest.json"}
        ROOT.mkdir()
        for name in names:
            member = ROOT / name
            member.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(name) as source, member.open("xb") as destination:
                import shutil
                shutil.copyfileobj(source, destination)
assert file_sha(ROOT / "manifest.json") == MANIFEST_SHA
# Check vendored source hashes before the first import, even on a resumed session.
manifest = json.loads((ROOT / "manifest.json").read_text())
for name, info in manifest["files"].items():
    path = ROOT / name
    assert not path.is_symlink() and file_sha(path) == info["sha256"], name
sys.path.insert(0, str(ROOT))
from src.vision import resnet50_fold0_pilot_v1 as pilot
PREFLIGHT = pilot.preflight(ROOT, MANIFEST_SHA)
print(json.dumps(PREFLIGHT, indent=2))


In [ ]:
# GPU/runtime check: no fitting occurs in this cell.
import torch, torchvision, numpy, sklearn, PIL
print({"torch": torch.__version__, "torchvision": torchvision.__version__,
       "numpy": numpy.__version__, "sklearn": sklearn.__version__, "Pillow": PIL.__version__})
assert torch.cuda.is_available(), "CUDA GPU required; stop instead of CPU training"
print(torch.cuda.get_device_name(), "CUDA", torch.version.cuda)


## Explicit future execution gate
After a separate owner approval, set RUN_TRAINING=True and OWNER_APPROVAL to:
`I approve this exact fold-0 Colab GPU pilot and cloud processing.`

Budget: shared warm-up 1 epoch, arm A 5 epochs, arm B 5 epochs; no early stopping,
score-driven selection or automatic extra epochs. Fresh Adam per stage; same arm batch orders.
Checkpoints are saved every epoch. Identical-runtime interrupted fits can resume with the
same RUN_ROOT. An incomplete checkpoint is preserved and fails closed. A completed run refuses
execution. Scoring/CAM export interruptions require inspection; they are not silently overwritten.


In [ ]:
if not CLOUD_UPLOAD_APPROVED or not RUN_TRAINING:
    raise PermissionError("Preparation complete. STOP and obtain explicit training approval.")
RESULT = pilot.run(ROOT, RUN_ROOT, execute=RUN_TRAINING,
                   approval=OWNER_APPROVAL, manifest_sha=MANIFEST_SHA)
print(json.dumps(RESULT, indent=2))


## Review and return results — only after a completed approved run
Review `metrics.json`, `predictions.json`, `warmup/`, `A/`, `B/` and `cam/`.
CAM contains predicted-class and common-A-target original/heatmap/overlay views for the same
eight predeclared photos. Attention is not validated disease localization or gum boundaries.
Doctor review should be blinded to model identity; record relevant/irrelevant/uncertain and
notes separately. No attention-mask score is claimed without qualified clinical annotation.

The optional return ZIP contains experimental CAM images as well as weights/results.
It stays on the Colab VM until you explicitly download it. It never installs into the PP1 demo.
Colab local storage is ephemeral: manually download checkpoints/results before ending the session.


In [ ]:
# Explicit optional packaging; never upload or install into the Mac demo automatically.
assert (RUN_ROOT / "complete.json").is_file(), "No completed experiment yet"
import shutil
RETURN_BASE = Path("/content/resnet50_fold0_return_v1")
assert not RETURN_BASE.with_suffix(".zip").exists(), "Preserve existing return ZIP"
files = {path.relative_to(RUN_ROOT).as_posix(): file_sha(path)
         for path in RUN_ROOT.rglob("*") if path.is_file()}
integrity = RUN_ROOT / "return_integrity.json"
assert not integrity.exists(), "Preserve existing integrity receipt"
integrity.write_text(json.dumps(files, indent=2))
shutil.make_archive(str(RETURN_BASE), "zip", RUN_ROOT)
print("Download manually:", RETURN_BASE.with_suffix(".zip"))
print("SHA-256:", file_sha(RETURN_BASE.with_suffix(".zip")))
